In [6]:
import torch
import torch.nn as nn
import math

In [9]:
embed_size = 64
qvk_dim = 64

to_query = nn.Linear(embed_size, qvk_dim)
to_key = nn.Linear(embed_size, qvk_dim)
to_value = nn.Linear(embed_size, qvk_dim)

input_sequence = torch.rand(1, 5, embed_size) # batch, sequence length, embed size

Query = to_query(input_sequence)
Key = to_key(input_sequence)
Value = to_value(input_sequence)

print(Query.shape)
print(Key.shape)
print(Value.shape)

torch.Size([1, 5, 64])
torch.Size([1, 5, 64])
torch.Size([1, 5, 64])


In [12]:
attention_score = torch.einsum('bqd, bkd -> bqk', Query, Value)
scale_factor = math.sqrt(qvk_dim)
scaled_attention = attention_score / scale_factor
attention_weights = torch.softmax(scaled_attention, dim=-1)
print(attention_weights)

tensor([[[0.1971, 0.1912, 0.2009, 0.1750, 0.2358],
         [0.2013, 0.1960, 0.1996, 0.1762, 0.2270],
         [0.2100, 0.1988, 0.1915, 0.1720, 0.2277],
         [0.2123, 0.1988, 0.1987, 0.1682, 0.2220],
         [0.2012, 0.1993, 0.1899, 0.1852, 0.2245]]],
       grad_fn=<SoftmaxBackward0>)


In [15]:
value_ = torch.einsum('bss, bsv -> bsv', attention_weights, Value)
print(value_.shape)

torch.Size([1, 5, 64])


In [22]:
embed_size = 64
qvk_dim = 64
head_num = 8
seq_len = 5
head_dim = qvk_dim // head_num

to_query = nn.Linear(embed_size, qvk_dim)
to_key = nn.Linear(embed_size, qvk_dim)
to_value = nn.Linear(embed_size, qvk_dim)

input_sequence = torch.rand(1, 5, embed_size) # batch, sequence length, embed size

Query = to_query(input_sequence)
Key = to_key(input_sequence)
Value = to_value(input_sequence)

print(Query.shape)
print(Key.shape)
print(Value.shape)

torch.Size([1, 5, 64])
torch.Size([1, 5, 64])
torch.Size([1, 5, 64])


In [28]:
Query = Query.view(-1, seq_len, head_num, head_dim)
Key = Key.view(-1, seq_len, head_num, head_dim)
Value = Value.view(-1, seq_len, head_num, head_dim)

print(Query.shape)
print(Key.shape)
print(Value.shape)

torch.Size([1, 5, 8, 8])
torch.Size([1, 5, 8, 8])
torch.Size([1, 5, 8, 8])


In [31]:
Query = Query.transpose(2, 1)
Key = Key.transpose(2, 1)
Value = Value.transpose(2, 1)
print(Query.shape)
print(Key.shape)
print(Value.shape)

torch.Size([1, 8, 5, 8])
torch.Size([1, 8, 5, 8])
torch.Size([1, 8, 5, 8])


In [35]:
multi_head_attention = torch.einsum('bhqd, bhkd -> bhqk', Query, Key)
print(multi_head_attention.shape)

torch.Size([1, 8, 5, 5])


In [36]:
scale_factor = math.sqrt(head_dim)
scaled_multi_head_attention = multi_head_attention / scale_factor
print(scaled_multi_head_attention.shape)

torch.Size([1, 8, 5, 5])


In [37]:
attention_multi_head_weights = torch.softmax(scaled_multi_head_attention, dim=-1)
print(attention_multi_head_weights.shape)

torch.Size([1, 8, 5, 5])


In [41]:
output = torch.einsum('bhss, bhsv -> bhsv', attention_multi_head_weights, Value)
print(value_.shape)
output = output.transpose(1, 2)
print(output.shape)
output = output.contiguous().view(1, seq_len, -1)
print(output.shape)

torch.Size([1, 8, 5, 8])
torch.Size([1, 5, 8, 8])
torch.Size([1, 5, 64])
